# Foundations & Estimation

## What's covered

- What "system design" actually means, and why it's a different discipline from writing code
- **Scalability** — vertical vs horizontal, and the state-management cliff between them
- **Availability, reliability, and durability** — three properties that get confused constantly
- **Latency vs throughput** — and why tail latency (p99) is what users actually feel
- **Back-of-envelope estimation** — the math that anchors every later decision
- The **latency numbers every engineer should know** — the table you'll reference for the rest of the series


## What system design is

System design is the discipline of designing the *structure and behaviour* of a software system that has to serve real users at real scale. It is what changes when "the code works on my laptop" stops being the bar.

A useful framing: when you write a function, you optimize for **correctness**. When you write a service, you optimize for **fitness** — the system meets its performance, availability, cost, and operability targets under realistic load and realistic failure. Fitness is plural. There is no single right answer. Every decision is a trade-off between competing goals, and the job is to make the trade-offs deliberately rather than by accident.

The vocabulary in this notebook — scalability, availability, latency, capacity — is the language we use to *describe* those trade-offs precisely. The rest of the series uses this vocabulary constantly. If "availability" and "reliability" both mean "the system works" to you right now, this notebook is where we sharpen them.

A short note on style: system design is heavy on **numbers** and light on **code**. You'll see capacity math in every chapter — bytes per record, requests per second, latency budget, fan-out factor. The numbers anchor decisions. "Use a CDN" is a recipe. "Use a CDN because the round-trip from Mumbai to us-east-1 is 200ms and our p99 budget is 100ms" is a decision.


## Scalability — vertical vs horizontal

When load grows, you have two options.

**Vertical scaling** (also called *scaling up*) — make the existing machine bigger. More CPU cores, more RAM, a faster disk. The code does not change. The deployment does not change. You just provision a larger instance.

**Horizontal scaling** (also called *scaling out*) — add more machines. Run the same code on each. Spread requests across them with a load balancer.

```
   Vertical                          Horizontal
   ========                          ==========

   +-----+         +-------------+       +---+ +---+ +---+
   |     |   -->   |             |       | M | | M | | M |
   |  M  |         |      M      |       +---+ +---+ +---+
   |     |         |             |         ^     ^     ^
   +-----+         +-------------+         |     |     |
                                          +--+ load +--+
                                             | balancer |
                                             +----------+
```

Vertical is the path of least resistance. No code changes, no distributed-systems problems. You hit the ceiling fast though — the biggest cloud instance is finite, and the price-per-unit-of-capacity climbs steeply at the top end. Disk speed plateaus. Cross-NUMA memory access doesn't scale linearly. And every workload that runs on one machine has a single failure point — when that machine reboots, your service is down.

Horizontal scaling has no ceiling but pays a price: **state becomes the hard problem**. Stateless services scale horizontally cheaply — every request can land on any machine. Stateful services (your database, your cache, your session store) need to share state somehow, and the rest of this series is partly the story of how. Sharding, replication, consensus, eventual consistency — all of these exist to support horizontal scale of stateful systems.

The pragmatic rule of thumb: **scale vertically until it hurts, then scale horizontally**. A small service running on a four-core VM at 30% CPU should not be sharded — you would inherit distributed-systems complexity for no benefit. A service whose CPU pegs at peak load should add machines, not size them up indefinitely.


## Three S's — availability, reliability, durability

These three are confused constantly. They measure different things.

**Availability** is the percentage of time the system *responds* to requests. A service that is up but returning HTTP 500 to every request is *not* available. A service that is up and returning 200s to every request is fully available.

Common availability targets are written in "nines":

| Nines | Availability | Downtime per year |
|---|---|---|
| 2 nines | 99% | 3.65 days |
| 3 nines | 99.9% | 8.77 hours |
| 4 nines | 99.99% | 52.6 minutes |
| 5 nines | 99.999% | 5.26 minutes |

Five nines is the standard target for telecom-grade infrastructure. Most web services target three or four. *Each additional nine costs roughly an order of magnitude more*, because the cheap failures (single VM crash, deploy gone wrong) are handled at three nines; the rare failures (datacenter outage, network partition) only matter at four-plus.

**Reliability** is the probability that the system *behaves correctly* over a given time window. Availability is "the system responded." Reliability is "the response was right." A service that incorrectly applies a $100 charge to ten thousand customers is *available* (it responded) but *unreliable* (it was wrong). Reliability is measured by MTBF (mean time between failures) or by error rate.

**Durability** is the probability that *data* survives over a given time window. S3 famously advertises eleven nines of durability — 99.999999999% — meaning if you store ten million objects, on average you'd lose one every ten thousand years. Durability is achieved by replication and erasure coding; the more redundant copies, the higher the durability.

You can have any combination. A service can be highly available but unreliable (responds fast, gives wrong answers), or reliable but unavailable (gives perfect answers, but only when it's up — and it's not up). A database can be highly durable (your data never disappears) but low-availability (the database might be unreachable while the data sits safely on disk).


## Latency vs throughput

Two properties of any pipeline. Often confused. They are not the same.

**Latency** is the time a single request takes from start to finish. Measured in milliseconds. Smaller is better.

**Throughput** is the number of requests the system can handle per unit time. Measured in requests per second (RPS or QPS) or in bytes per second. Larger is better.

The highway analogy makes the distinction concrete. Latency is the time for one car to drive from A to B — say, an hour at 60 mph for 60 miles. Throughput is how many cars per hour the highway delivers — say, a thousand cars per hour at full capacity. **Adding lanes raises throughput without changing the single-car latency.** Raising the speed limit lowers latency without changing the throughput (if cars are spaced the same). They are two independent dials.

For a real service, the relationship is subtler. As you push more requests per second through a fixed capacity, individual requests start queueing — they wait their turn for CPU, for a connection, for a disk seek. **Adding capacity often raises throughput at the cost of latency**, because each request has more queueing in front of it. Conversely, parallelizing a single request across many workers lowers its latency at the cost of throughput, because the workers are now spread thin.

The two questions to ask about any system: *how fast is a single request* (latency) and *how many can we handle at once* (throughput). Most performance work targets one or the other, not both at once.


## Tail latency — p50, p99, p99.9

The phrase "the system has 100ms latency" is almost always a lie of omission. What does "100ms latency" mean — the average? The median? The slowest 1%?

Latency is distributed, not single-valued. You report it as **percentiles**:

- **p50** (median) — half of requests are faster than this, half are slower.
- **p95** — 95% of requests are faster; the slowest 5% are slower.
- **p99** — 99% of requests are faster; one in a hundred is slower.
- **p99.9** ("three-nines latency") — one in a thousand is slower than this.

Why percentiles instead of averages? Because **latency distributions are heavy-tailed**. A small number of slow requests can drag the mean way above the median. A service with a 10ms median latency might have a 200ms p99 — the *typical* request is fast, but one in a hundred is twenty times slower.

```
  count
   |
   |    ###
   |   #####                                                     <- median is here
   |   #######                                                       (typical case)
   |   ##########
   |   #############
   |    ##############
   |     ##############
   |       ##############
   |          ##############
   |             #################
   |                  ########################                   <- p99 is way over here
   |                        ################################         (tail is heavy)
   +--+--+--+--+--+--+--+--+--+--+--+--+--+--+--+--+--+--+--+--+--+--+----> latency
      10ms                                                          200ms
```

The high percentiles are what users feel. If your service makes ten downstream calls to render a single page, the page's latency is roughly the *worst* of those ten calls — which is well-approximated by p90 of any one call. Ten calls means the user effectively experiences the p90 of every call. Make a hundred calls and you're effectively experiencing the p99 every time.

**Tail latency is what your system actually delivers to the slowest 1% of users — and that 1% is loud, repeat customers, and the ones writing support tickets.** Most modern services define their SLOs (Service Level Objectives) in terms of p99 or p99.9, not average.


## Back-of-envelope estimation

Before designing a system, do the math. A few quick calculations rule out architectures that cannot possibly work, validate that simpler ones might, and anchor the decisions that follow.

**The four numbers you usually need:**

- **Active users** — daily or monthly. Twitter has ~250M DAU. A startup has 50.
- **Action rate** — requests per active user per day. A social-feed user might generate 500 requests per day across all their browsing.
- **Payload size** — bytes per request, bytes per record. A tweet is ~280 bytes raw; a tweet object with metadata is ~1-2KB.
- **Time window** — over a day, a month, a year.

Multiply through. **Storage: users × records-per-user × bytes-per-record.** **Bandwidth: requests-per-second × bytes-per-response.** **QPS: total requests ÷ seconds in the window.**

Two shortcuts that save constant arithmetic:

- **86,400 seconds in a day**, often rounded to **~100,000** for quick math.
- **One million requests per day ≈ 12 QPS.** Useful for translating "we have 100M requests a day" into "1,200 QPS average". Peak QPS is typically 2-3x average.

A worked example: a chat app with 10 million daily active users, each sending 20 messages per day at an average of 200 bytes per message.

- Daily messages: 10M × 20 = 200 million per day.
- Average QPS: 200M ÷ 86,400 ≈ 2,300 QPS.
- Peak QPS (rule of thumb 3x): ~7,000 QPS.
- Daily ingest: 200M × 200B = 40 GB per day.
- One year: 40 GB × 365 ≈ 14.6 TB.
- Five years: ~73 TB.

That single calculation rules out architectures that store every message in memory (too expensive) and validates that a sharded relational database with cold-storage offload is plausible. We haven't designed anything yet, but we've narrowed the search.


### Capacity math, in code


In [ ]:
# A small set of helpers for back-of-envelope estimation.

SECONDS_PER_DAY  = 86_400
DAYS_PER_YEAR    = 365
KB = 1_000
MB = 1_000_000
GB = 1_000_000_000
TB = 1_000_000_000_000


def daily_qps(daily_users: int, actions_per_user: int) -> float:
    return daily_users * actions_per_user / SECONDS_PER_DAY


def peak_qps(avg_qps: float, peak_multiplier: float = 3.0) -> float:
    return avg_qps * peak_multiplier


def daily_bytes(daily_users: int, actions_per_user: int, bytes_per_action: int) -> int:
    return daily_users * actions_per_user * bytes_per_action


def yearly_bytes(daily_users: int, actions_per_user: int, bytes_per_action: int) -> int:
    return daily_bytes(daily_users, actions_per_user, bytes_per_action) * DAYS_PER_YEAR


def humanize_bytes(n: int) -> str:
    for unit, size in [("TB", TB), ("GB", GB), ("MB", MB), ("KB", KB)]:
        if n >= size:
            return f"{n / size:.2f} {unit}"
    return f"{n} B"


# Worked example — chat app.
DAU = 10_000_000          # 10M daily active users
MSGS_PER_USER = 20        # 20 messages per user per day
BYTES_PER_MSG = 200       # avg message size including metadata

avg = daily_qps(DAU, MSGS_PER_USER)
peak = peak_qps(avg)
per_day = daily_bytes(DAU, MSGS_PER_USER, BYTES_PER_MSG)
per_year = yearly_bytes(DAU, MSGS_PER_USER, BYTES_PER_MSG)

print(f"average QPS:  {avg:>10,.0f}")
print(f"peak QPS:     {peak:>10,.0f}")
print(f"per day:      {humanize_bytes(per_day):>10}")
print(f"per year:     {humanize_bytes(per_year):>10}")
print(f"5-year storage: {humanize_bytes(per_year * 5)}")


## Latency numbers every engineer should know

The single most useful table in system design. These are the numbers Jeff Dean famously published in 2010, updated here for 2025-era hardware. **Internalize the *orders of magnitude*** — the exact values shift year to year, but the ratios are stable.

| Operation | Time | Notes |
|---|---|---|
| L1 cache reference | 1 ns | per core, on-die |
| Branch mispredict | 3 ns | pipeline stall |
| L2 cache reference | 4 ns | on-die |
| Mutex lock/unlock | 17 ns | uncontended |
| **Main memory reference** | **100 ns** | DRAM, after L3 miss |
| Compress 1KB with snappy | 2,000 ns (2 μs) | per kilobyte |
| Send 1KB over 1 Gbps network | 10 μs | wire time only |
| **Read 4KB from SSD** | **150 μs** | NVMe random read |
| Read 1MB sequentially from memory | 3 μs | streaming read |
| Read 1MB sequentially from SSD | 1 ms | NVMe |
| **Round-trip in same datacenter** | **500 μs (0.5 ms)** | host-to-host within DC |
| Read 1MB sequentially from disk (HDD) | 20 ms | spinning disk |
| Disk seek (HDD) | 10 ms | spinning disk |
| **TCP packet across continent (US)** | **40 ms** | NY to SF round-trip |
| **TCP packet across the world** | **150 ms** | Mumbai to us-east-1 round-trip |

A few rules of thumb that fall out of this table:

- **L1 to memory is 100x.** A cache miss costs the time of a hundred cache hits.
- **Memory to SSD is 1000x.** A disk read costs the time of a thousand memory reads.
- **SSD to disk is 100x.** Spinning disks are dead for hot data; SSDs everywhere.
- **Same-DC to cross-continent is 100x.** Geography is the dominant latency cost on the open internet.
- **Cross-continent to cross-world is 4x.** Once you've left your datacenter, every hop is expensive.

These numbers explain *why* caching exists (RAM is 1000x faster than SSD), *why* CDNs exist (cross-continent round-trips are unaffordable for static content), *why* connection pooling matters (TLS handshakes cost multiple round-trips), and *why* batching wins (per-call overhead dominates for small operations).


## Designing for the numbers

A few examples of how the latency table shapes architecture:

**Why caching exists.** A web request that hits the database makes a 0.5-2ms call. The same request served from an in-process cache is ~100ns — five orders of magnitude faster. Even a Redis cache on the same network (sub-millisecond) is 5-10x faster than a SQL query. Caching trades correctness (cache may be stale) for latency. The decision to cache anything is the decision to accept some staleness in exchange for that 1000x speedup.

**Why CDNs exist.** A user in Mumbai hitting a server in Virginia pays 150ms just for the network round-trip — before any compute happens. A CDN with a point of presence in Mumbai serves the cached content in 5-10ms. For static assets (images, JavaScript, CSS), the staleness is acceptable and the latency win is enormous.

**Why batching wins.** If you need to read 100 keys from a database, doing 100 separate queries costs roughly 100 × (round-trip + query time). Doing one query with `WHERE key IN (...)` costs roughly one round-trip plus the database's batch-processing time. The round-trip cost is usually 10-100x larger than the per-key compute cost, so batching is almost always a win.

**Why connection pooling matters.** A new TCP connection costs three round-trips (SYN, SYN-ACK, ACK) before any data flows. A new TLS connection costs another round-trip or two for the handshake. On a cross-DC link, that's 200ms+ of overhead per request *just to open the connection*. A connection pool reuses live connections, amortizing that cost across thousands of requests.

The pattern: when you see a system-design recommendation that doesn't make sense on its face, the latency table usually explains it.


## Capacity planning vs load testing

Two ways to know whether your system will hold up. Both are necessary.

**Capacity planning** is the forward calculation. Start from expected traffic, multiply through the dependency graph, and arrive at required hardware: "We expect 10K QPS peak, each request fans out to 3 service calls, each call needs 50ms of CPU, so we need 10K × 3 × 0.05 = 1,500 CPU-seconds per second, divided by 0.7 utilization target, so about 21 cores." Capacity planning catches *architectural* problems — fan-out factors you forgot, payloads that are larger than budgeted, indexes you'll need at scale.

**Load testing** is the empirical measurement. Run real load against a real (or near-real) system and observe what breaks. Load testing catches *operational* problems — connection limits, thread-pool starvation, GC pauses, the specific moment when latency goes from acceptable to terrible.

The two complement each other. Capacity planning is forward-looking; it tells you what to build. Load testing is backward-validating; it tells you whether what you built actually behaves as planned. Skip capacity planning and you design the wrong system. Skip load testing and you ship the system without knowing where it breaks.

A common mistake is to load-test only the steady state. The interesting failures happen at the *edges* — when load suddenly doubles, when one dependency starts slowing down, when a single hot key floods one shard. Load tests should include traffic spikes, dependency degradation, and skewed key distributions, not just constant uniform load.


Notebook two turns to **Networking & Edge** — the path a request takes from the user's browser to your servers and back. DNS resolution. CDNs. Load balancers at layer four and layer seven. REST versus gee-are-pee-see versus graph-q-l. Rate limiting. These are the components that sit between your users and your services, and the latency numbers from this notebook are the budget every one of them has to fit into.
